
## Folder 05 — File 03: Data Type and Format Validation:


**Purpose**: Validate whether Silver records contain correctly typed and consistently formatted values.

This notebook checks:

1. Email address formats.
2. Phone number formats.
3. Date and timestamp validity.
4. Numeric data types.
5. Allowed categorical values.
6. Percentage and decimal ranges.
7. Currency codes and identifier formats.

**Important distinction:** A column having the correct Spark data type does not mean its values are valid. For example, a string column can contain an invalid email address.


## Important technical notes:


- Phone validation is intentionally basic. A number can match the pattern and still be unassigned or invalid. Real phone verification requires a proper numbering library or verification service.
- Currency validation does not establish that a currency is supported by your business. The three-letter pattern validates formatting. If your project is INR-only, add this additional business rule:
- F.col("currency") != "INR"
- Check the product stock statuses. The allowed values in this notebook assume IN_STOCK, OUT_OF_STOCK, and LOW_STOCK. If your generated dataset uses different labels, adjust the allowed list to match the actual business definitions.
- Decimal types matter. Financial values should use DECIMAL, not floating-point types, to avoid unexpected rounding differences.
- This notebook reports failures but does not quarantine them. Quarantine handling is covered in a separate notebook.


## What is Quarantine in Data Engineering?


**Quarantine** is a `temporary storage area` where we isolate `bad, invalid, or suspicious records` instead of allowing them into the **clean Silver layer**.

In [0]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

from datetime import datetime

print("Libraries imported successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

REPORT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "data_type_format_report"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "quarantine_customers"
)

print("Project configuration completed.")


# ============================================================
# 3. EXPECTED DATA TYPES
# ============================================================

EXPECTED_TYPES = {

    "silver_customers": {
        "customer_id": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "phone": "string",
        "city": "string",
        "state": "string",
        "created_at": "date"
    },

    "silver_products": {
        "product_id": "string",
        "product_name": "string",
        "category": "string",
        "unit_price": "decimal(12,2)",
        "unit_cost": "decimal(12,2)",
        "stock_status": "string",
        "created_at": "date"
    },

    "silver_orders": {
        "order_id": "string",
        "customer_id": "string",
        "order_timestamp": "timestamp",
        "order_status": "string",
        "currency": "string",
        "order_total": "decimal(12,2)",
        "sales_channel": "string"
    },

    "silver_order_items": {
        "order_item_id": "string",
        "order_id": "string",
        "product_id": "string",
        "quantity": "int",
        "unit_price": "decimal(12,2)",
        "discount_rate": "decimal(5,4)",
        "line_total": "decimal(12,2)"
    },

    "silver_payments": {
        "payment_id": "string",
        "order_id": "string",
        "payment_method": "string",
        "payment_status": "string",
        "amount": "decimal(12,2)",
        "payment_timestamp": "timestamp"
    },

    "silver_shipments": {
        "shipment_id": "string",
        "order_id": "string",
        "carrier": "string",
        "shipment_status": "string",
        "shipped_date": "timestamp",
        "delivered_date": "timestamp",
        "tracking_number": "string"
    },

    "silver_inventory": {
        "inventory_snapshot_id": "string",
        "product_id": "string",
        "warehouse_id": "string",
        "quantity_on_hand": "int",
        "snapshot_date": "date"
    }
}


# ============================================================
# 4. ALLOWED CATEGORICAL VALUES
# ============================================================

ALLOWED_VALUES = {

    "silver_orders": {
        "order_status": [
            "DELIVERED",
            "SHIPPED",
            "PROCESSING",
            "CANCELLED"
        ]
    },

    "silver_products": {
        "stock_status": [
            "IN_STOCK",
            "OUT_OF_STOCK",
            "LOW_STOCK"
        ]
    },

    "silver_payments": {
        "payment_status": [
            "SUCCESS",
            "PENDING",
            "FAILED",
            "REFUNDED"
        ]
    },

    "silver_shipments": {
        "shipment_status": [
            "PENDING",
            "PROCESSING",
            "SHIPPED",
            "IN_TRANSIT",
            "OUT_FOR_DELIVERY",
            "DELIVERED",
            "CANCELLED",
            "RETURNED"
        ]
    }
}


# Normalize common shipment status formatting variations.
# These are aliases, not additional business statuses.

SHIPMENT_STATUS_ALIASES = {
    "IN TRANSIT": "IN_TRANSIT",
    "IN-TRANSIT": "IN_TRANSIT",
    "OUT FOR DELIVERY": "OUT_FOR_DELIVERY",
    "OUT-FOR-DELIVERY": "OUT_FOR_DELIVERY"
}


print("Validation configuration completed.")


# ============================================================
# 5. VALIDATION REPORT FUNCTIONS
# ============================================================

results = []


def add_result(
    dataset_name,
    check_name,
    column_name,
    total_records,
    failed_records,
    details
):

    total_records = int(total_records)
    failed_records = int(failed_records)

    failure_percentage = (
        round(failed_records / total_records * 100, 2)
        if total_records > 0
        else 0.0
    )

    status = (
        "PASS"
        if total_records > 0 and failed_records == 0
        else "FAIL"
    )

    results.append({

        "dataset_name": dataset_name,
        "check_name": check_name,
        "column_name": column_name,
        "total_records": total_records,
        "failed_records": failed_records,
        "failure_percentage": float(failure_percentage),
        "status": status,
        "details": str(details),
        "checked_at": datetime.now()
    })


def count_failed_records(df, condition):

    return df.filter(condition).count()


def get_silver_table(dataset_name):

    table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{dataset_name}"
    )

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Silver table not found: {table_name}"
        )

    return spark.table(table_name)


def validate_required_columns(
    df,
    dataset_name,
    required_columns
):

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise Exception(
            f"Missing columns in {dataset_name}: "
            f"{missing_columns}"
        )


def normalize_phone(column_name):

    return F.regexp_replace(
        F.trim(F.col(column_name)),
        r"[\s()\-]",
        ""
    )


def normalize_category(column_name):

    return F.upper(
        F.trim(F.col(column_name))
    )


print("Validation functions created.")


# ============================================================
# 6. VALIDATE SPARK DATA TYPES
# ============================================================

for dataset_name, expected_columns in EXPECTED_TYPES.items():

    df = get_silver_table(dataset_name)

    total_records = df.count()

    actual_types = {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }

    for column_name, expected_type in expected_columns.items():

        if column_name not in actual_types:

            add_result(
                dataset_name,
                "data_type_validation",
                column_name,
                total_records,
                1,
                f"Missing column: {column_name}"
            )

            continue

        actual_type = actual_types[column_name]

        failed = int(actual_type != expected_type)

        add_result(
            dataset_name,
            "data_type_validation",
            column_name,
            total_records,
            failed,
            (
                f"Expected: {expected_type}; "
                f"Actual: {actual_type}"
            )
        )

print("Data type validation completed.")


# ============================================================
# 7. CUSTOMER EMAIL FORMAT VALIDATION
# ============================================================

customers_df = get_silver_table("silver_customers")

validate_required_columns(
    customers_df,
    "silver_customers",
    ["email", "phone"]
)

customer_count = customers_df.count()

email_pattern = (
    r"^[A-Za-z0-9._%+-]+@"
    r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)

invalid_email_condition = (
    F.col("email").isNotNull()
    & (
        ~F.trim(F.col("email")).rlike(email_pattern)
    )
)

invalid_email_count = count_failed_records(
    customers_df,
    invalid_email_condition
)

add_result(
    "silver_customers",
    "email_format_validation",
    "email",
    customer_count,
    invalid_email_count,
    "Non-null email values must match the configured email pattern."
)

print("Email validation completed.")


# ============================================================
# 8. CUSTOMER PHONE FORMAT VALIDATION
# ============================================================

phone_pattern = r"^\+?[0-9]{10,15}$"

normalized_customer_phone = normalize_phone("phone")

invalid_phone_condition = (
    F.col("phone").isNotNull()
    & (
        ~normalized_customer_phone.rlike(phone_pattern)
    )
)

invalid_phone_count = count_failed_records(
    customers_df,
    invalid_phone_condition
)

add_result(
    "silver_customers",
    "phone_format_validation",
    "phone",
    customer_count,
    invalid_phone_count,
    (
        "Non-null phone values must contain 10-15 digits, "
        "optionally prefixed with +, after formatting normalization."
    )
)

print("Phone validation completed.")

# ------------------------------------------------------------
# 9.1 ALLOWED VALUES CONFIGURATION
# ------------------------------------------------------------

ALLOWED_VALUES = {
    "silver_orders": {
        "order_status": [
            "PENDING",
            "PROCESSING",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED",
            "RETURNED",
            "REFUNDED"
        ]
    },

    "silver_payments": {
        "payment_status": [
            "PENDING",
            "SUCCESS",
            "FAILED",
            "REFUNDED"
        ]
    },

    "silver_shipments": {
        "shipment_status": [
            "PENDING",
            "PROCESSING",
            "SHIPPED",
            "IN_TRANSIT",
            "OUT_FOR_DELIVERY",
            "DELIVERED",
            "CANCELLED",
            "RETURNED",
            "RETURN_TO_ORIGIN"
        ]
    }
}


# ------------------------------------------------------------
# 9.2 STATUS ALIASES
# ------------------------------------------------------------

SHIPMENT_STATUS_ALIASES = {
    "IN TRANSIT": "IN_TRANSIT",
    "IN-TRANSIT": "IN_TRANSIT",

    "OUT FOR DELIVERY": "OUT_FOR_DELIVERY",
    "OUT-FOR-DELIVERY": "OUT_FOR_DELIVERY",

    "RETURN TO ORIGIN": "RETURN_TO_ORIGIN",
    "RETURN-TO-ORIGIN": "RETURN_TO_ORIGIN"
}


# ------------------------------------------------------------
# 9.3 VALIDATE CATEGORICAL VALUES
# ------------------------------------------------------------

for dataset_name, columns in ALLOWED_VALUES.items():

    df = get_silver_table(dataset_name)

    total_records = df.count()

    for column_name, allowed_values in columns.items():

        # Verify that the column exists.
        validate_required_columns(
            df,
            dataset_name,
            [column_name]
        )

        # Normalize case and whitespace.
        normalized_value = F.upper(
            F.trim(F.col(column_name))
        )

        # Apply shipment status aliases.
        if (
            dataset_name == "silver_shipments"
            and column_name == "shipment_status"
        ):

            status_mapping = F.create_map(
                *[
                    value
                    for pair in SHIPMENT_STATUS_ALIASES.items()
                    for value in (
                        F.lit(pair[0]),
                        F.lit(pair[1])
                    )
                ]
            )

            normalized_value = F.coalesce(
                F.element_at(
                    status_mapping,
                    normalized_value
                ),
                normalized_value
            )

        # Identify null and disallowed values.
        invalid_condition = (
            F.col(column_name).isNull()
            | (~normalized_value.isin(allowed_values))
        )

        failed_count = count_failed_records(
            df,
            invalid_condition
        )

        # Save validation result.
        add_result(
            dataset_name,
            "allowed_category_validation",
            column_name,
            total_records,
            failed_count,
            (
                f"Allowed normalized values: {allowed_values}. "
                "Whitespace, letter case and configured aliases "
                "are normalized."
            )
        )

        print(
            f"{dataset_name}.{column_name}: "
            f"{total_records - failed_count} passed, "
            f"{failed_count} failed"
        )


print("Categorical value validation completed.")



# ============================================================
# 10. CURRENCY CODE VALIDATION
# ============================================================

orders_df = get_silver_table("silver_orders")

validate_required_columns(
    orders_df,
    "silver_orders",
    ["currency"]
)

order_count = orders_df.count()

invalid_currency_condition = (
    F.col("currency").isNull()
    | (
        ~F.trim(F.col("currency")).rlike(r"^[A-Z]{3}$")
    )
)

invalid_currency_count = count_failed_records(
    orders_df,
    invalid_currency_condition
)

add_result(
    "silver_orders",
    "currency_format_validation",
    "currency",
    order_count,
    invalid_currency_count,
    "Currency must be a three-letter uppercase code."
)

print("Currency validation completed.")


# ============================================================
# 11. NUMERIC BUSINESS RULE VALIDATION
# ============================================================

numeric_rules = {

    "silver_products": [
        ("unit_price", lambda: F.col("unit_price") < 0),
        ("unit_cost", lambda: F.col("unit_cost") < 0)
    ],

    "silver_orders": [
        ("order_total", lambda: F.col("order_total") < 0)
    ],

    "silver_order_items": [
        ("quantity", lambda: F.col("quantity") <= 0),
        ("unit_price", lambda: F.col("unit_price") < 0),
        (
            "discount_rate",
            lambda: (
                (F.col("discount_rate") < 0)
                | (F.col("discount_rate") > 1)
            )
        ),
        ("line_total", lambda: F.col("line_total") < 0)
    ],

    "silver_payments": [
        ("amount", lambda: F.col("amount") < 0)
    ],

    "silver_inventory": [
        ("quantity_on_hand", lambda: F.col("quantity_on_hand") < 0)
    ]
}


for dataset_name, rules in numeric_rules.items():

    df = get_silver_table(dataset_name)

    total_records = df.count()

    for column_name, condition_function in rules:

        validate_required_columns(
            df,
            dataset_name,
            [column_name]
        )

        failed_count = count_failed_records(
            df,
            condition_function()
        )

        add_result(
            dataset_name,
            "numeric_business_rule",
            column_name,
            total_records,
            failed_count,
            f"Invalid numeric values found in {column_name}."
        )

print("Numeric business rule validation completed.")


# ============================================================
# 12. ORDER ITEM ARITHMETIC VALIDATION
# ============================================================

order_items_df = get_silver_table("silver_order_items")

validate_required_columns(
    order_items_df,
    "silver_order_items",
    [
        "quantity",
        "unit_price",
        "discount_rate",
        "line_total"
    ]
)

item_count = order_items_df.count()

expected_line_total = (
    F.col("quantity").cast("decimal(18,4)")
    * F.col("unit_price").cast("decimal(18,4)")
    * (
        F.lit(1).cast("decimal(18,4)")
        - F.col("discount_rate").cast("decimal(18,4)")
    )
)

difference = (
    F.col("line_total").cast("decimal(18,4)")
    - expected_line_total
)

invalid_line_total_condition = (
    F.col("quantity").isNotNull()
    & F.col("unit_price").isNotNull()
    & F.col("discount_rate").isNotNull()
    & F.col("line_total").isNotNull()
    & (
        (difference > F.lit(0.02))
        | (difference < F.lit(-0.02))
    )
)

invalid_line_total_count = count_failed_records(
    order_items_df,
    invalid_line_total_condition
)

add_result(
    "silver_order_items",
    "line_total_reconciliation",
    "line_total",
    item_count,
    invalid_line_total_count,
    (
        "Line total must match quantity * unit price * "
        "(1 - discount rate), within 0.02."
    )
)

print("Order item arithmetic validation completed.")


# ============================================================
# 13. SHIPMENT DATE CONSISTENCY VALIDATION
# ============================================================

shipments_df = get_silver_table("silver_shipments")

validate_required_columns(
    shipments_df,
    "silver_shipments",
    [
        "shipment_status",
        "shipped_date",
        "delivered_date"
    ]
)

shipment_count = shipments_df.count()

missing_delivery_date = (
    (F.upper(F.trim(F.col("shipment_status"))) == "DELIVERED")
    & F.col("delivered_date").isNull()
)

delivery_without_shipment_date = (
    F.col("delivered_date").isNotNull()
    & F.col("shipped_date").isNull()
)

delivery_before_shipment = (
    F.col("delivered_date").isNotNull()
    & F.col("shipped_date").isNotNull()
    & (
        F.col("delivered_date") < F.col("shipped_date")
    )
)

invalid_shipment_dates = (
    missing_delivery_date
    | delivery_without_shipment_date
    | delivery_before_shipment
)

invalid_shipment_date_count = count_failed_records(
    shipments_df,
    invalid_shipment_dates
)

add_result(
    "silver_shipments",
    "shipment_date_consistency",
    "delivered_date",
    shipment_count,
    invalid_shipment_date_count,
    (
        "Delivered shipments require a delivery date. "
        "Delivery cannot exist without shipment date "
        "or precede shipment."
    )
)

print("Shipment date validation completed.")


# ============================================================
# 14. CUSTOMER QUARANTINE VALIDATION
# ============================================================

if not spark.catalog.tableExists(QUARANTINE_TABLE):
    raise Exception(
        f"Customer quarantine table not found: {QUARANTINE_TABLE}"
    )

quarantine_df = spark.table(QUARANTINE_TABLE)

quarantine_count = quarantine_df.count()

# Support both common quarantine reason column names.
if "quarantine_reason" in quarantine_df.columns:
    reason_column = "quarantine_reason"
elif "_quarantine_reason" in quarantine_df.columns:
    reason_column = "_quarantine_reason"
else:
    raise Exception(
        "Neither quarantine_reason nor _quarantine_reason "
        "exists in quarantine_customers."
    )

validate_required_columns(
    quarantine_df,
    "quarantine_customers",
    ["customer_id", "phone", reason_column]
)

# Check 1: Every quarantine record must have a reason.

missing_reason_condition = (
    F.col(reason_column).isNull()
    | (F.trim(F.col(reason_column)) == "")
)

missing_reason_count = count_failed_records(
    quarantine_df,
    missing_reason_condition
)

add_result(
    "quarantine_customers",
    "quarantine_reason_validation",
    reason_column,
    quarantine_count,
    missing_reason_count,
    "Every quarantined record must have a non-empty reason."
)

# Check 2: Check phone validity only when the quarantine
# reason specifically identifies a phone-related problem.

phone_reason_condition = (
    F.lower(F.col(reason_column)).contains("phone")
)

normalized_quarantine_phone = normalize_phone("phone")

valid_phone_for_phone_quarantine = (
    F.col("phone").isNotNull()
    & normalized_quarantine_phone.rlike(phone_pattern)
)

unexpected_valid_phone_count = count_failed_records(
    quarantine_df,
    phone_reason_condition
    & valid_phone_for_phone_quarantine
)

phone_reason_record_count = count_failed_records(
    quarantine_df,
    phone_reason_condition
)

if phone_reason_record_count > 0:

    add_result(
        "quarantine_customers",
        "quarantine_phone_validation",
        "phone",
        phone_reason_record_count,
        unexpected_valid_phone_count,
        (
            "Records quarantined specifically for phone issues "
            "must contain an invalid or missing phone number."
        )
    )

else:

    add_result(
        "quarantine_customers",
        "quarantine_phone_validation",
        "phone",
        quarantine_count,
        0,
        (
            "No phone-specific quarantine reasons were found. "
            "Phone validity is not assumed for unrelated quarantine records."
        )
    )

# Check 3: Duplicate customer IDs in quarantine.

duplicate_quarantine_ids = (
    quarantine_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("customer_id").isNotNull()
        & (F.col("count") > 1)
    )
    .count()
)

add_result(
    "quarantine_customers",
    "quarantine_duplicate_validation",
    "customer_id",
    quarantine_count,
    duplicate_quarantine_ids,
    "Customer IDs must not have duplicate groups in quarantine."
)

print("Customer quarantine validation completed.")


# ============================================================
# 15. VERIFY INVALID PHONES ARE NOT IN SILVER
# ============================================================

# This check deliberately uses the same normalized phone
# rule as the phone format validation.

invalid_phone_in_silver_count = count_failed_records(
    customers_df,
    invalid_phone_condition
)

add_result(
    "silver_customers",
    "invalid_phone_quarantine_verification",
    "phone",
    customer_count,
    invalid_phone_in_silver_count,
    (
        "Invalid non-null phone values must not remain in "
        "Silver customers. Correct them or quarantine them."
    )
)

print("Silver phone consistency validation completed.")


# ============================================================
# 16. CREATE VALIDATION REPORT
# ============================================================

report_schema = StructType([

    StructField("dataset_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("column_name", StringType(), True),

    StructField("total_records", LongType(), False),
    StructField("failed_records", LongType(), False),

    StructField("failure_percentage", DoubleType(), False),

    StructField("status", StringType(), False),
    StructField("details", StringType(), True),

    StructField("checked_at", TimestampType(), False)
])

if not results:
    raise Exception("No validation results were generated.")

report_df = spark.createDataFrame(
    results,
    schema=report_schema
)

print("Validation report created.")

display(
    report_df.orderBy(
        "dataset_name",
        "check_name",
        "column_name"
    )
)


# ============================================================
# 17. SAVE REPORT TO DELTA TABLE
# ============================================================

(
    report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REPORT_TABLE)
)

print(f"Report saved: {REPORT_TABLE}")


# ============================================================
# 18. DATASET-LEVEL VALIDATION SUMMARY
# ============================================================

summary_df = (
    report_df
    .groupBy("dataset_name")
    .agg(
        F.count("*").alias("total_checks"),

        F.sum(
            F.when(F.col("status") == "PASS", 1).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(F.col("status") == "FAIL", 1).otherwise(0)
        ).alias("failed_checks")
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_checks") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
    .orderBy("dataset_name")
)

display(summary_df)


# ============================================================
# 19. DISPLAY FAILED CHECKS
# ============================================================

failed_report_df = (
    report_df
    .filter(F.col("status") == "FAIL")
    .orderBy(
        "dataset_name",
        "check_name",
        "column_name"
    )
)

print("FAILED VALIDATION CHECKS")

display(failed_report_df)


# ============================================================
# 20. FINAL VALIDATION SUMMARY
# ============================================================

total_checks = report_df.count()

passed_checks = (
    report_df
    .filter(F.col("status") == "PASS")
    .count()
)

failed_checks = (
    report_df
    .filter(F.col("status") == "FAIL")
    .count()
)

print("=" * 65)
print("DATA TYPE, FORMAT AND QUARANTINE VALIDATION SUMMARY")
print("=" * 65)

print(f"Total checks  : {total_checks}")
print(f"Passed checks : {passed_checks}")
print(f"Failed checks : {failed_checks}")

print("=" * 65)

if failed_checks == 0:
    print("RESULT: ALL VALIDATION CHECKS PASSED.")
else:
    print(
        f"RESULT: {failed_checks} VALIDATION CHECK(S) FAILED."
    )

print()
print("CUSTOMER RECORD SUMMARY")
print(f"Silver customers     : {customer_count}")
print(f"Quarantined customers: {quarantine_count}")

print()
print("REPORT TABLE")
print(REPORT_TABLE)

print("=" * 65)


# ============================================================
# 21. DISPLAY FINAL SAVED REPORT
# ============================================================

display(
    spark.table(REPORT_TABLE)
    .orderBy(
        "dataset_name",
        "check_name",
        "column_name"
    )
)

print("Data type, format and quarantine validation completed.")


### Identify the exact failures

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_silver.data_type_format_report
WHERE UPPER(status) = 'FAIL'
ORDER BY dataset_name, check_name;


**Verify the shipment status independently**

In [0]:
%sql

SELECT
    shipment_status,
    COUNT(*) AS record_count
FROM workspace.ecommerce_silver.silver_shipments
GROUP BY shipment_status
ORDER BY record_count DESC;


**inspect the shipment statuses separately:**

In [0]:
%sql

SELECT
    shipment_status,
    COUNT(*) AS record_count
FROM workspace.ecommerce_silver.silver_shipments
GROUP BY shipment_status
ORDER BY record_count DESC;


**inspect invalid customer phone numbers:**

In [0]:
%sql

SELECT
    customer_id,
    phone
FROM workspace.ecommerce_silver.silver_customers
WHERE phone IS NOT NULL
  AND NOT (
      REGEXP_REPLACE(TRIM(phone), '[\\s()\\-]', '')
      RLIKE '^\\+?[0-9]{10,15}$'
  );


### Recover and append invalid phone records

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. TABLE CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.ecommerce_bronze.bronze_customers"
SILVER_TABLE = "workspace.ecommerce_silver.silver_customers"
QUARANTINE_TABLE = "workspace.ecommerce_silver.quarantine_customers"


# ============================================================
# 2. READ BRONZE AND QUARANTINE TABLES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE)

quarantine_df = spark.table(QUARANTINE_TABLE)

bronze_columns = bronze_df.columns

print("Bronze records:", bronze_df.count())
print("Existing quarantine records:", quarantine_df.count())


# ============================================================
# 3. IDENTIFY INVALID PHONE NUMBERS
# ============================================================

# Remove spaces, brackets and hyphens before validating.
normalized_phone = F.regexp_replace(
    F.col("phone"),
    r"[\s()\-]",
    ""
)

# Valid format:
# 10-15 digits, optionally starting with +

invalid_phone_condition = (
    F.col("phone").isNotNull()
    & ~normalized_phone.rlike(r"^\+?[0-9]{10,15}$")
)

invalid_records = bronze_df.filter(invalid_phone_condition)

print("Invalid phone records found in Bronze:", invalid_records.count())

display(
    invalid_records.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone"
    )
)


# ============================================================
# 4. REMOVE RECORDS ALREADY PRESENT IN QUARANTINE
# ============================================================

existing_quarantine_ids = (
    quarantine_df
    .select("customer_id")
    .filter(F.col("customer_id").isNotNull())
    .distinct()
)

new_records = invalid_records.join(
    existing_quarantine_ids,
    on="customer_id",
    how="left_anti"
)

print("New records to quarantine:", new_records.count())


# ============================================================
# 5. ADD QUARANTINE INFORMATION
# ============================================================

new_records = (
    new_records
    .withColumn(
        "quarantine_reason",
        F.lit("INVALID_PHONE_FORMAT")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)


# ============================================================
# 6. ALIGN DATAFRAME WITH EXISTING TABLE SCHEMA
# ============================================================

target_schema = quarantine_df.schema

aligned_expressions = []

for field in target_schema.fields:

    column_name = field.name

    # Map Bronze ingestion timestamp to quarantine timestamp.
    if column_name == "_ingestion_timestamp":

        if "_ingested_at" in bronze_columns:
            expression = F.col("_ingested_at").cast(field.dataType)
        elif "_ingestion_timestamp" in bronze_columns:
            expression = F.col("_ingestion_timestamp").cast(field.dataType)
        else:
            expression = F.lit(None).cast(field.dataType)

    # Derive ingestion date if necessary.
    elif column_name == "_ingestion_date":

        if "_ingestion_date" in bronze_columns:
            expression = F.col("_ingestion_date").cast(field.dataType)
        elif "_ingested_at" in bronze_columns:
            expression = F.to_date("_ingested_at").cast(field.dataType)
        elif "_ingestion_timestamp" in bronze_columns:
            expression = F.to_date("_ingestion_timestamp").cast(field.dataType)
        else:
            expression = F.lit(None).cast(field.dataType)

    # Preserve source columns that exist.
    elif column_name in new_records.columns:

        expression = F.col(column_name).cast(field.dataType)

    # Fill missing target columns with typed NULL.
    else:

        expression = F.lit(None).cast(field.dataType)

    aligned_expressions.append(
        expression.alias(column_name)
    )


aligned_records = new_records.select(*aligned_expressions)


# ============================================================
# 7. APPEND TO QUARANTINE TABLE
# ============================================================

records_to_insert = aligned_records.count()

if records_to_insert > 0:

    (
        aligned_records.write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )

    print("Quarantine append completed.")

else:

    print("No new invalid phone records to append.")


# ============================================================
# 8. VERIFY QUARANTINE TABLE
# ============================================================

updated_quarantine = spark.table(QUARANTINE_TABLE)

print("Total quarantined customer records:",
      updated_quarantine.count())

display(
    updated_quarantine.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone",
        "quarantine_reason",
        "quarantined_at"
    )
)


### Verify the three quarantine conditions:


After the append succeeds, run:

In [0]:
from pyspark.sql import functions as F

quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_customers"
)

silver_df = spark.table(
    "workspace.ecommerce_silver.silver_customers"
)

# 1. Check duplicate customer IDs
duplicate_ids = (
    quarantine_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate quarantine IDs:", duplicate_ids.count())


# 2. Check invalid phone numbers
normalized_phone = F.regexp_replace(
    F.col("phone"),
    r"[\s()\-]",
    ""
)

valid_phone_condition = normalized_phone.rlike(
    r"^\+?[0-9]{10,15}$"
)

valid_quarantine_phones = quarantine_df.filter(
    F.col("phone").isNull() | valid_phone_condition
)

print("Invalid quarantine phone records:",
      quarantine_df.count() - valid_quarantine_phones.count())


# 3. Check quarantine reasons
missing_reasons = quarantine_df.filter(
    F.col("quarantine_reason").isNull()
    | (F.trim(F.col("quarantine_reason")) == "")
)

print("Records with missing quarantine reasons:",
      missing_reasons.count())


# 4. Check invalid phones remaining in Silver
silver_invalid_phones = silver_df.filter(
    F.col("phone").isNotNull()
    & ~F.regexp_replace(
        F.col("phone"),
        r"[\s()\-]",
        ""
    ).rlike(r"^\+?[0-9]{10,15}$")
)

print("Invalid phone records remaining in Silver:",
      silver_invalid_phones.count())


# 5. Final counts
print("Silver customers:", silver_df.count())
print("Quarantined customers:", quarantine_df.count())


### Move invalid customers into quarantine:

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = (
    "workspace.ecommerce_silver.silver_customers"
)

QUARANTINE_TABLE = (
    "workspace.ecommerce_silver.quarantine_customers"
)

customers_df = spark.table(SILVER_TABLE)

# Identify invalid phone numbers
phone_pattern = r"^\+?[0-9]{10,15}$"

normalized_phone = F.regexp_replace(
    F.trim(F.col("phone")),
    r"[\s()\-]",
    ""
)

invalid_condition = (
    F.col("phone").isNotNull()
    & (~normalized_phone.rlike(phone_pattern))
)

# Separate valid and invalid records
valid_customers = customers_df.filter(
    ~invalid_condition
)

quarantined_customers = (
    customers_df
    .filter(invalid_condition)
    .withColumn(
        "quarantine_reason",
        F.lit("Invalid phone number format")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

# Display counts before writing
print("Original customers:", customers_df.count())
print("Valid customers:", valid_customers.count())
print("Quarantined customers:", quarantined_customers.count())

# Save valid customers
(
    valid_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# Save quarantined customers
(
    quarantined_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Customer phone quarantine completed.")


### Verify quarantine records:

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "workspace.ecommerce_silver.silver_customers"
QUARANTINE_TABLE = "workspace.ecommerce_silver.quarantine_customers"

silver_df = spark.table(SILVER_TABLE)

print("Silver customer count:", silver_df.count())

print("Quarantine table exists:",
      spark.catalog.tableExists(QUARANTINE_TABLE))

if spark.catalog.tableExists(QUARANTINE_TABLE):

    quarantine_df = spark.table(QUARANTINE_TABLE)

    print("Quarantine record count:", quarantine_df.count())

    display(quarantine_df)


### Check whether the invalid records are still in Silver:

In [0]:
phone_pattern = r"^\+?[0-9]{10,15}$"

normalized_phone = F.regexp_replace(
    F.trim(F.col("phone")),
    r"[\s()\-]",
    ""
)

invalid_condition = (
    F.col("phone").isNotNull()
    & (~normalized_phone.rlike(phone_pattern))
)

invalid_records = silver_df.filter(invalid_condition)

print("Invalid phone records still in Silver:",
      invalid_records.count())

display(
    invalid_records.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone"
    )
)


### Recover the three records from Bronze:

In [0]:
from pyspark.sql import functions as F

BRONZE_PATH = (
    "/Volumes/workspace/default/data_volume/"
    "ecommerce_lakehouse/bronze/customers"
)

QUARANTINE_TABLE = (
    "workspace.ecommerce_silver.quarantine_customers"
)

# Read original Bronze data
bronze_customers = (
    spark.read
    .format("delta")
    .load(BRONZE_PATH)
)

# Identify the three affected customers
customer_ids = [
    "CUST000212",
    "CUST003211",
    "CUST008801"
]

recovered_records = (
    bronze_customers
    .filter(F.col("customer_id").isin(customer_ids))
    .withColumn(
        "quarantine_reason",
        F.lit("Invalid phone number format")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

print("Recovered records:", recovered_records.count())

display(
    recovered_records.select(
        "customer_id",
        "first_name",
        "last_name",
        "phone",
        "quarantine_reason"
    )
)

In [0]:

# Databricks notebook source

# MAGIC %md
# MAGIC # 07.5 - Silver Data Quality Remediation
# MAGIC
# MAGIC Remediate invalid customer phone numbers and update
# MAGIC the allowed shipment status values.

# COMMAND ----------

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Libraries imported successfully.")

# COMMAND ----------

# ============================================================
# 2. CONFIGURATION
# ============================================================

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

CUSTOMER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
)

CUSTOMER_QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_customers"
)

SHIPMENT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_shipments"
)

REMEDIATION_REPORT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.data_quality_remediation_report"
)

CUSTOMER_KEY = "customer_id"
PHONE_COLUMN = "phone"

EXPECTED_INVALID_CUSTOMERS = [
    "CUST000212",
    "CUST003211",
    "CUST008801"
]

ALLOWED_SHIPMENT_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN_TRANSIT",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "RETURNED",
    "RETURN_TO_ORIGIN"
]

print("Configuration loaded.")

# COMMAND ----------

# ============================================================
# 3. VALIDATE SOURCE TABLES
# ============================================================

required_tables = [
    CUSTOMER_TABLE,
    CUSTOMER_QUARANTINE_TABLE,
    SHIPMENT_TABLE
]

for table_name in required_tables:

    if not spark.catalog.tableExists(table_name):
        raise Exception(f"Required table not found: {table_name}")

print("All required tables exist.")

# COMMAND ----------

# ============================================================
# 4. IDENTIFY INVALID CUSTOMER PHONE NUMBERS
# ============================================================

customers_df = spark.table(CUSTOMER_TABLE)

required_columns = [
    CUSTOMER_KEY,
    PHONE_COLUMN
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in customers_df.columns
]

if missing_columns:
    raise Exception(
        f"Missing customer columns: {missing_columns}"
    )

# Remove spaces, parentheses and hyphens before validation.
normalized_phone = F.regexp_replace(
    F.trim(F.col(PHONE_COLUMN)),
    r"[\s()\-]",
    ""
)

invalid_phone_condition = (
    F.col(PHONE_COLUMN).isNotNull()
    & (~normalized_phone.rlike(r"^\+?[0-9]{10,15}$"))
)

invalid_customers_df = (
    customers_df
    .filter(invalid_phone_condition)
)

invalid_count = invalid_customers_df.count()

print("Invalid customer phone records:", invalid_count)

invalid_customers_df.select(
    CUSTOMER_KEY,
    PHONE_COLUMN
).show(truncate=False)

# COMMAND ----------

# ============================================================
# 5. VERIFY EXPECTED INVALID CUSTOMER RECORDS
# ============================================================

actual_invalid_ids = {
    row[CUSTOMER_KEY]
    for row in invalid_customers_df.select(CUSTOMER_KEY).collect()
}

expected_invalid_ids = set(EXPECTED_INVALID_CUSTOMERS)

if actual_invalid_ids != expected_invalid_ids:
    raise Exception(
        "Invalid customer IDs do not match the expected records.\n"
        f"Expected: {sorted(expected_invalid_ids)}\n"
        f"Actual: {sorted(actual_invalid_ids)}\n"
        "Stop and inspect the source records before remediation."
    )

print("Expected invalid customer IDs verified.")

# COMMAND ----------

# ============================================================
# 6. PREPARE CUSTOMER QUARANTINE TABLE
# ============================================================

quarantine_df = spark.table(CUSTOMER_QUARANTINE_TABLE)

# Identify the existing reason column.
reason_candidates = [
    "quarantine_reason",
    "_quarantine_reason",
    "reason"
]

reason_column = next(
    (
        column_name
        for column_name in reason_candidates
        if column_name in quarantine_df.columns
    ),
    None
)

# Add a reason column if the quarantine table does not have one.
if reason_column is None:

    spark.sql(
        f"""
        ALTER TABLE {CUSTOMER_QUARANTINE_TABLE}
        ADD COLUMNS (quarantine_reason STRING)
        """
    )

    reason_column = "quarantine_reason"

    quarantine_df = spark.table(CUSTOMER_QUARANTINE_TABLE)

print("Quarantine reason column:", reason_column)

# COMMAND ----------

# ============================================================
# 7. PREVENT DUPLICATE QUARANTINE INSERTS
# ============================================================

existing_quarantine_ids = (
    quarantine_df
    .select(CUSTOMER_KEY)
    .filter(F.col(CUSTOMER_KEY).isNotNull())
    .distinct()
)

new_quarantine_records = (
    invalid_customers_df
    .join(
        existing_quarantine_ids,
        on=CUSTOMER_KEY,
        how="left_anti"
    )
    .withColumn(
        reason_column,
        F.lit("INVALID_PHONE_FORMAT")
    )
)

new_quarantine_count = new_quarantine_records.count()

print(
    "New customer records to quarantine:",
    new_quarantine_count
)

# Align source columns to the existing quarantine table schema.
target_schema = spark.table(
    CUSTOMER_QUARANTINE_TABLE
).schema

source_columns = set(new_quarantine_records.columns)

aligned_records = new_quarantine_records

for field in target_schema.fields:

    if field.name not in source_columns:

        aligned_records = aligned_records.withColumn(
            field.name,
            F.lit(None).cast(field.dataType)
        )

aligned_records = aligned_records.select(
    *[
        F.col(field.name).cast(field.dataType)
        for field in target_schema.fields
    ]
)

# Append only new records.
if new_quarantine_count > 0:

    (
        aligned_records
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(CUSTOMER_QUARANTINE_TABLE)
    )

print("Customer quarantine insertion completed.")

# COMMAND ----------

# ============================================================
# 8. VERIFY QUARANTINE INSERTION
# ============================================================

updated_quarantine_df = spark.table(
    CUSTOMER_QUARANTINE_TABLE
)

quarantined_invalid_records = (
    updated_quarantine_df
    .filter(
        F.col(CUSTOMER_KEY).isin(EXPECTED_INVALID_CUSTOMERS)
    )
)

quarantined_invalid_count = (
    quarantined_invalid_records
    .select(CUSTOMER_KEY)
    .distinct()
    .count()
)

quarantined_invalid_records.select(
    CUSTOMER_KEY,
    PHONE_COLUMN,
    reason_column
).show(truncate=False)

if quarantined_invalid_count != len(EXPECTED_INVALID_CUSTOMERS):
    raise Exception(
        "Customer quarantine verification failed. "
        f"Expected {len(EXPECTED_INVALID_CUSTOMERS)}, "
        f"found {quarantined_invalid_count}."
    )

print("All three invalid customers exist in quarantine.")

# COMMAND ----------

# ============================================================
# 9. REMOVE INVALID CUSTOMERS FROM SILVER
# ============================================================

silver_before = spark.table(CUSTOMER_TABLE).count()

if invalid_count > 0:

    DeltaTable.forName(
        spark,
        CUSTOMER_TABLE
    ).delete(
        invalid_phone_condition
    )

silver_after = spark.table(CUSTOMER_TABLE).count()

deleted_count = silver_before - silver_after

print("Silver customers before:", silver_before)
print("Silver customers after :", silver_after)
print("Removed records        :", deleted_count)

# COMMAND ----------

# ============================================================
# 10. VERIFY CUSTOMER PHONE QUALITY
# ============================================================

remaining_invalid_phones = (
    spark.table(CUSTOMER_TABLE)
    .filter(
        F.col(PHONE_COLUMN).isNotNull()
        & (
            ~F.regexp_replace(
                F.trim(F.col(PHONE_COLUMN)),
                r"[\s()\-]",
                ""
            ).rlike(r"^\+?[0-9]{10,15}$")
        )
    )
    .count()
)

remaining_expected_ids = (
    spark.table(CUSTOMER_TABLE)
    .filter(
        F.col(CUSTOMER_KEY).isin(EXPECTED_INVALID_CUSTOMERS)
    )
    .count()
)

if remaining_invalid_phones != 0:
    raise Exception(
        f"{remaining_invalid_phones} invalid phone records remain in Silver."
    )

if remaining_expected_ids != 0:
    raise Exception(
        "One or more invalid customer IDs remain in Silver."
    )

print("Customer phone validation PASSED.")

# COMMAND ----------

# ============================================================
# 11. UPDATE SHIPMENT STATUS ALLOWED VALUES
# ============================================================

shipments_df = spark.table(SHIPMENT_TABLE)

if "shipment_status" not in shipments_df.columns:
    raise Exception(
        "shipment_status column not found in Silver shipments."
    )

shipment_status_counts = (
    shipments_df
    .groupBy("shipment_status")
    .count()
    .orderBy(F.desc("count"))
)

print("Shipment status distribution:")
shipment_status_counts.show(truncate=False)

# Normalize case and whitespace for validation.
normalized_status = F.upper(
    F.trim(F.col("shipment_status"))
)

invalid_status_condition = (
    F.col("shipment_status").isNull()
    | (~normalized_status.isin(ALLOWED_SHIPMENT_STATUSES))
)

invalid_shipment_status_count = (
    shipments_df
    .filter(invalid_status_condition)
    .count()
)

print(
    "Shipment records with disallowed statuses:",
    invalid_shipment_status_count
)

# COMMAND ----------

# ============================================================
# 12. VERIFY RETURN_TO_ORIGIN
# ============================================================

return_to_origin_count = (
    shipments_df
    .filter(
        F.upper(F.trim(F.col("shipment_status")))
        == "RETURN_TO_ORIGIN"
    )
    .count()
)

print(
    "RETURN_TO_ORIGIN records:",
    return_to_origin_count
)

if return_to_origin_count == 0:
    print(
        "WARNING: No RETURN_TO_ORIGIN records found. "
        "Check the source status distribution."
    )

if invalid_shipment_status_count != 0:
    raise Exception(
        "Shipment status validation still has invalid values. "
        "Inspect the status distribution before changing the rules."
    )

print("Shipment status validation PASSED.")

# COMMAND ----------

# ============================================================
# 13. SAVE REMEDIATION REPORT
# ============================================================

customer_pass = remaining_invalid_phones == 0
shipment_pass = invalid_shipment_status_count == 0

report_rows = [
    (
        "silver_customers",
        "invalid_phone_remediation",
        PHONE_COLUMN,
        int(invalid_count),
        int(deleted_count),
        "PASS" if customer_pass else "FAIL",
        "Invalid phone records quarantined and removed from Silver."
    ),
    (
        "silver_shipments",
        "allowed_category_validation",
        "shipment_status",
        int(shipments_df.count()),
        int(invalid_shipment_status_count),
        "PASS" if shipment_pass else "FAIL",
        "Allowed shipment statuses include RETURN_TO_ORIGIN."
    )
]

report_schema = [
    "dataset_name",
    "check_name",
    "column_name",
    "total_records",
    "failed_records",
    "status",
    "details"
]

report_df = spark.createDataFrame(
    report_rows,
    report_schema
).withColumn(
    "run_timestamp",
    F.current_timestamp()
)

(
    report_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REMEDIATION_REPORT_TABLE)
)

print("Remediation report saved:", REMEDIATION_REPORT_TABLE)

# COMMAND ----------

# ============================================================
# 14. FINAL SUMMARY
# ============================================================

final_silver_count = spark.table(CUSTOMER_TABLE).count()
final_quarantine_count = spark.table(
    CUSTOMER_QUARANTINE_TABLE
).count()

final_status = (
    "PASSED"
    if customer_pass and shipment_pass
    else "FAILED"
)

print("=" * 65)
print("SILVER DATA QUALITY REMEDIATION SUMMARY")
print("=" * 65)

print("Invalid customer phones found :", invalid_count)
print("New quarantine records added  :", new_quarantine_count)
print("Customer records removed      :", deleted_count)
print("Remaining invalid phones      :", remaining_invalid_phones)

print("-" * 65)

print("Silver customers after        :", final_silver_count)
print("Customer quarantine after     :", final_quarantine_count)

print("-" * 65)

print("RETURN_TO_ORIGIN records       :", return_to_origin_count)
print("Invalid shipment statuses     :", invalid_shipment_status_count)

print("-" * 65)

print("Remediation result            :", final_status)
print("Remediation report             :", REMEDIATION_REPORT_TABLE)

print("=" * 65)

if final_status != "PASSED":
    raise Exception("One or more remediation checks failed.")

print("SILVER DATA QUALITY REMEDIATION COMPLETED.")
